# 10 · Production run: biogenic emissions for any region and period

This notebook computes the emission inventory described by a run configuration (`config/*.yaml`): the grid, the
region, the period, the land-cover source and the MEGAN input tables. For every month of the period it samples
ERA5-Land, MODIS LAI and the land cover in Google Earth Engine on the model grid, runs the MEGANv3.21 engine of
`megan_sc` (canopy, activity factors, soil NO, speciation) every `sample_hours` hours, and writes the monthly and
annual emission grids, the inputs as sampled and a log of each procedure. The input and run procedures live in
`src/megan_sc/fixes.py`, which this notebook writes into the repository folder (cell 2); its SHA-256 is stamped in
every output file, so each output can be traced to the code that produced it.

| Procedure (label in `fixes.py`) | What it does |
|---|---|
| FIX-1 | cells without ERA5-Land data (the reanalysis covers land only) take all meteorological variables from the nearest valid land cell (one donor for all bands); the run stops if a vegetated cell keeps invalid meteorology |
| FIX-2 | D_TEMP = 24-h mean 2 m temperature of the UTC day (24 hourly ERA5-Land records, as daymet.f); D_PPFD from the day's PAR |
| FIX-3 | leaf age with the previous month's LAI as LAIp (NLAI = 12) |
| FIX-4 | monthly MODIS LAI gaps filled from the adjacent valid months |
| FIX-5 | no-data pixels of the MEGAN ecotype map take the nearest valid ecotype |
| FIX-6 | GAMTP applies the canopy-depth factor to the sunlit term only, as megvea.f does |
| FIX-7 | air temperature capped at 303 K before the soil-temperature proxy, as soilnox.F does |
| FIX-8 | ERA5-Land radiation de-accumulated: previous hour subtracted at every hour except 01 UTC (ECMWF ERA5-Land convention; the 00 UTC field holds the previous day's 24-h total) |

Settings that follow the MEGANv3.21 code or are fixed by the configuration: the soil-NO growing season of each
latitude band (`soil.growseason`), constant light-dependent fractions for the monoterpene classes, a constant wilting
point (`datasets.soil.constant_wilt`), the canopy type of trees (`datasets.vegetation.tree_canopy_type`), and the
soil-moisture and bidirectional-exchange factors on.

**Configurations.** `config/santa_catarina_2003_2025.yaml` (default) reproduces the Santa Catarina example run shipped
in `data/run_v10`. For another region, `run_your_region/region_setup.ipynb` writes `config/<region>_run.yaml`; set
`RUN_CONFIG` to that file. The tutorial (`docs/tutorial/06_run_the_model.md`) explains every setting.

**How to run.** Set `RUN_CONFIG` and `EE_PROJECT` (cell 3), optionally restrict `YEARS`, and run all cells.
Finished months are skipped and an interrupted month resumes from its checkpoint, so after a Colab disconnect just
run all cells again. Outputs go to `OUT_BASE/<output.dir of the configuration>`. When the run finishes, run
`11_verify_run.ipynb` (checks) and `25_results_any_region.ipynb` (tables, maps, trends) with the same `RUN_CONFIG`.

In [ ]:
# 1 · Setup ---------------------------------------------------------------------------------------------
# --- Find the MEGAN_SC folder and, in Google Colab, connect Google Drive (the same block opens every notebook) ---------
import os, sys, glob
try:
    from google.colab import drive as _drive
    IN_COLAB = 'MEGAN_SC_RUNNER' not in os.environ          # tools/run_notebook.py sets it when it runs a notebook outside Colab
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    # A failed mount must stop the notebook: otherwise it would read and write a local /content/drive folder that is not
    # Google Drive.
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive') and os.listdir('/content/drive'):
        raise RuntimeError('/content/drive holds local files written while Google Drive was not mounted, so Drive cannot be '
                           'mounted there. Use Runtime > Disconnect and delete runtime, then run the notebook again.')
    _drive.mount('/content/drive')


def find_repository():
    """The MEGAN_SC folder: $MEGAN_SC_REPO when set; in Colab, MyDrive/MEGAN_SC (or a MEGAN_SC folder one level down in
    MyDrive, or /content/MEGAN_SC for a clone outside Drive); elsewhere, the current folder or its parent."""
    cands = [os.environ.get('MEGAN_SC_REPO', '')]
    if IN_COLAB:
        cands += ['/content/drive/MyDrive/MEGAN_SC', '/content/drive/MyDrive/megan_sc', '/content/drive/MyDrive/megan-sc']
        cands += sorted(glob.glob('/content/drive/MyDrive/*/MEGAN_SC')) + ['/content/MEGAN_SC']
    cands += [os.getcwd(), os.path.dirname(os.getcwd())]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, 'src', 'megan_sc', 'fixes.py')):
            return os.path.abspath(c)
    raise SystemExit('The MEGAN_SC folder was not found. In Colab, copy the repository to Google Drive as MyDrive/MEGAN_SC '
                     '(tutorial, chapter 3), or run  os.environ["MEGAN_SC_REPO"] = "/path/to/MEGAN_SC"  before this cell.')


REPO_DIR = find_repository()
if os.path.join(REPO_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('MEGAN_SC folder:', REPO_DIR, '| Colab:', IN_COLAB)

%pip -q install earthengine-api numpy scipy xarray netCDF4 matplotlib pyyaml pandas rasterio geopandas
import json, time, datetime, calendar, logging, hashlib, re, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
MEGAN_DIR = REPO_DIR                                    # the repository folder found above
for _lg in ('google_auth_httplib2', 'googleapiclient.http', 'google.auth'):
    logging.getLogger(_lg).setLevel(logging.ERROR)

In [ ]:
# 2 · Write megan_sc/fixes.py (the input and run procedures; src/megan_sc/fixes.py of the repository, 12 unit tests) --
FIXES_PY = r'''"""megan_sc.fixes -- corrections to the MEGAN-SC production run (adversarial review, Oct 2026).

Copy this file to  megan-sc/src/megan_sc/fixes.py .  It does not modify any existing module; the
corrected production cells (colab_cells_v10.py) call `run_timestep_fixed` instead of
`run.run_timestep`.  Every change is traceable to a MEGANv3.21 source line or to a diagnosed
artefact in the delivered 2003-2025 output.

FIX-1  Coastal / no-data meteorology (CRITICAL).
       ERA5-Land is land-only.  At cells it does not cover, Earth Engine returns a non-finite or a
       very large fill value.  `run.run_timestep` sanitised these with nan_to_num (PAR -> 0) and a
       clip of temperature to [150, 340] K, assuming such cells carry LAI = 0.  In the delivered
       run, 120 coastal cells inside Santa Catarina (821 km2) carry MODIS LAI and MapBiomas
       vegetation, so they were run every hour with T = 340 K and PAR = 0: ISOP, MBO and CO (LDF = 1)
       were exactly 0 and the light-independent part of every other class was multiplied by
       exp(beta*(340-303.15)) (x40 for monoterpenes, x525 for sesquiterpenes).  This produced
       16-22 % of the state BVOC total and ~75 % of the sesquiterpenes in every year.
       Fix: replace every invalid met value by the value of the nearest valid cell (all bands taken
       from the same donor cell) BEFORE the engine runs, and refuse to run (`strict=True`) if a
       vegetated cell is still invalid.
FIX-2  Daily-mean meteorology (MAJOR).
       `run.run_timestep` defaults to D_TEMP = T(hour) - 2 K and D_PPFD = 0.5*PPFD(hour) when the
       caller passes none; the production notebook passed none.  In MEGANv3.21 D_TEMP and D_PPFD
       are the 24-h means of the simulation day (daymet.f L287-320; read by megvea.f L515-523) and
       D_TEMP enters GAMTLD through Eopt = Cleo*exp(0.1*(T24-297)) and Topt (megvea.f GAMTLD).
       Fix: `run_timestep_fixed` REQUIRES d_temp and d_ppfd; the Colab cells compute D_TEMP as the
       mean of the 24 hourly ERA5-Land 2 m temperatures of the UTC day and D_PPFD from the day's PAR.
FIX-3  Leaf age (MODERATE).
       `run.run_timestep` called megvea with LAIp = LAIc and NLAI = 46, which pins GAMMA_A to its
       steady-state branch.  MEGANv3.21 reads the previous LAI period (megvea.f L585-600, FINDLAI).
       Fix: pass the previous month's LAI and NLAI = 12 (TSTLEN = 30 d).
FIX-5  Ecotype no-data on coastal land (MINOR).
       The MEGAN v3 ecotype map (EVT) carries the no-data code 65535 on a thin coastal strip.  Vegetated
       analysis cells covered only by no-data get EF = 0 and emit nothing (28 SC cells, 192 km2,
       ~0.15 % of state BVOC; e.g. the whole municipality of Governador Celso Ramos).
       Fix: `fill_ecotype_nearest` writes a copy of the ecotype file in which no-data pixels take the
       nearest valid ecotype; point the config's emission_factors.ecotype to that copy.
FIX-4  MODIS LAI monthly gaps (MINOR).
       Vegetated cells with no good-quality MCD15A3H retrieval in a month received LAI = 0 and
       therefore zero emission (0-1.4 % of annual isoprene, concentrated in cloudy summer months).
       Fix: `fill_lai_gaps` fills such cell-months from the adjacent valid months.
FIX-6  Canopy-depth factor in GAMTP (MAJOR for fidelity; found by the Fortran benchmark, Oct 2026).
       megvea.f (L725-731) computes
           Ea1L(K) = CDEA(K)*GAMTLD(SunT)*GAMP(SunP)*SunF + GAMTLD(ShaT)*GAMP(ShaP)*(1-SunF)
       i.e. by Fortran operator precedence CDEA multiplies ONLY the sunlit term.  The Python port
       (gamma.gamtp) multiplied both terms by CDEA.  On the official SE_US test case this changed the
       light-dependent ER by up to 25 % per cell-hour (+2 % domain total).  Fix: `gamtp_megan321`
       reproduces the reference code exactly (benchmark: max relative difference 4e-6).
FIX-7  Soil-NO air-temperature path (MODERATE; found by the Fortran benchmark).
       soilnox.F caps TAIR at 303 K BEFORE both the grass factor and the soil-temperature proxy
       TSOI = 0.72*TAIR + 82.28 (no-soil path).  The port capped only the grass branch, so crop soil NO
       was too high whenever T2 > 30 C.  Fix: `soilnox_megan321` applies the cap as soilnox.F does.
FIX-8  ERA5-Land radiation de-accumulation at 00 and 01 UTC (MODERATE; found by the notebook test, Oct 2026).
       ERA5-Land accumulations run from 00 UTC, and the field valid at 00 UTC holds the 24 h ending at
       00 UTC (ECMWF, ERA5-Land data documentation).  gee/era5.hourly_flux did not subtract anything at
       00 UTC (so PAR at 00 UTC was the previous day's 24-h total / 3600 x 0.45, ~2700 W m-2 in summer) and
       subtracted the 00 UTC field at 01 UTC (PAR clipped to 0).  In SC 00 UTC is night (about 21 h local
       solar time), so leaf light stayed 0, but the canopy stability/lapse term saw the spurious radiation
       (+4 to +8 % light-independent ER in that sample) and the PAR maps/driver series were inflated.
       Fix: `hourly_flux_fixed` subtracts the previous hour always except at 01 UTC.
NOTE   Southern-Hemisphere growing season (deliberate, documented deviation, kept).
       growseason.f compares the day of year (1-366) with MMDD constants (1101, 0101, 0531) in the
       lat < -23 branch, so south of 23 S the Fortran growing season becomes DOY 101-366.  The port's
       soil.growseason(corrected=True) applies the intended 1 Nov - 31 May season.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage

from .engine import canopy, megvea, soil, speciation as spc, gamma as _gamma, species as _sp

MET_KEYS = ("temp", "par", "wind", "pres", "qv", "soilm")
# physically admissible ranges for the ERA5-Land-derived MEGAN fields (SI units used by the engine)
VALID_RANGE = {
    "temp": (150.0, 340.0),       # K (2 m air temperature)
    "par": (0.0, 1000.0),         # W m-2 (0.45 x shortwave down)
    "wind": (0.0, 75.0),          # m s-1
    "pres": (4.0e4, 1.1e5),       # Pa
    "qv": (0.0, 0.06),            # kg kg-1
    "soilm": (0.0, 1.0),          # m3 m-3
}


# --------------------------------------------------------------------------- FIX-1
def invalid_met_mask(met: dict) -> np.ndarray:
    """True where ANY engine met band is non-finite or outside its physical range."""
    bad = None
    for k in MET_KEYS:
        if k not in met or met[k] is None:
            continue
        x = np.asarray(met[k], dtype=np.float64)
        lo, hi = VALID_RANGE[k]
        b = ~np.isfinite(x) | (x < lo) | (x > hi)
        bad = b if bad is None else (bad | b)
    if bad is None:
        raise ValueError("met dict has none of the expected keys " + str(MET_KEYS))
    return bad


def nearest_valid_index(bad: np.ndarray):
    """(ii, jj) index arrays pointing every cell to its nearest valid cell (itself if valid)."""
    if bad.all():
        raise RuntimeError("FIX-1: no valid meteorology anywhere on the grid")
    _, (ii, jj) = ndimage.distance_transform_edt(bad, return_indices=True)
    return ii, jj


def fill_met_nearest(met: dict, bad: np.ndarray | None = None, keys=MET_KEYS) -> tuple[dict, np.ndarray]:
    """Return (filled_met, bad_mask).  All bands of an invalid cell come from ONE donor cell."""
    bad = invalid_met_mask(met) if bad is None else bad
    out = dict(met)
    if not bad.any():
        return out, bad
    ii, jj = nearest_valid_index(bad)
    for k in keys:
        if k not in met or met[k] is None:
            continue
        x = np.array(met[k], dtype=np.float64, copy=True)
        x[bad] = x[ii[bad], jj[bad]]
        out[k] = x
    return out, bad


def fill_field_nearest(field, lo=-np.inf, hi=np.inf) -> np.ndarray:
    """Nearest-valid fill for a single 2-D field (e.g. daily D_TEMP)."""
    x = np.array(field, dtype=np.float64, copy=True)
    bad = ~np.isfinite(x) | (x < lo) | (x > hi)
    if bad.any():
        ii, jj = nearest_valid_index(bad)
        x[bad] = x[ii[bad], jj[bad]]
    return x


def assert_valid_on_vegetation(met: dict, lai) -> None:
    """Raise if any cell with LAI > 0 still has invalid meteorology (guards against FIX-1 regressions)."""
    lai = np.nan_to_num(np.asarray(lai, dtype=np.float64), nan=0.0)
    bad = invalid_met_mask(met) & (lai > 0)
    if bad.any():
        raise RuntimeError(f"FIX-1: {int(bad.sum())} vegetated cells still have invalid meteorology")


# --------------------------------------------------------------------------- FIX-4
def fill_lai_gaps(lai_m, lai_prev, lai_next, veg_frac, veg_min=0.05) -> tuple[np.ndarray, np.ndarray]:
    """Fill vegetated cells whose monthly LAI is missing/zero from the adjacent valid months.

    Uses the mean of the previous and next month where both are valid, otherwise whichever is
    valid.  Cells without vegetation (veg_frac <= veg_min) are left untouched.  Returns
    (filled_lai, filled_mask)."""
    cur = np.nan_to_num(np.asarray(lai_m, dtype=np.float64), nan=0.0)
    p = np.nan_to_num(np.asarray(lai_prev, dtype=np.float64), nan=0.0)
    n = np.nan_to_num(np.asarray(lai_next, dtype=np.float64), nan=0.0)
    gap = (cur <= 0) & (np.asarray(veg_frac) > veg_min) & ((p > 0) | (n > 0))
    both = (p > 0) & (n > 0)
    repl = np.where(both, 0.5 * (p + n), np.where(p > 0, p, n))
    out = np.where(gap, repl, cur)
    return out, gap


# --------------------------------------------------------------------------- FIX-2 helpers
PAR_WM2_TO_PPFD = 4.5      # daymet.f L320 / megcan.f


def daily_means_from_samples(temp_samples, par_samples):
    """D_TEMP (K) and D_PPFD (umol m-2 s-1) from a stack of same-day samples (time on axis 0).

    Prefer a 24-hourly stack for D_TEMP (MEGAN daymet uses 24 records); with 3-hourly samples the
    mean of the 8 values is an unbiased estimator of the daily mean for a smooth diurnal cycle.
    D_PPFD only acts as a night gate in MEGANv3.21 (C1 is constant in GAMP)."""
    t = np.asarray(temp_samples, dtype=np.float64)
    p = np.asarray(par_samples, dtype=np.float64)
    return t.mean(axis=0), p.mean(axis=0) * PAR_WM2_TO_PPFD


# --------------------------------------------------------------------------- corrected timestep
def run_timestep_fixed(domain, ef, met, *, day, hour_utc, canopy_type_fractions,
                       d_temp, d_ppfd, laip, nlai=12, wilt=0.12, date_yyyyddd=None,
                       soiltype=None, strict=True):
    """Drop-in replacement for run.run_timestep with FIX-1, FIX-2 and FIX-3 applied.

    `met` holds temp, par, wind, pres, qv, soilm, lai (current-month LAI, already gap-filled).
    `d_temp`, `d_ppfd` are the daily means of the simulation day (required).
    `laip` is the previous month's LAI (required); `nlai` = 12 for monthly LAI.
    Returns the same dict as run.run_timestep plus 'filled' (cells whose met was replaced).
    """
    if d_temp is None or d_ppfd is None:
        raise ValueError("FIX-2: d_temp and d_ppfd (daily means of the simulation day) are required")
    if laip is None:
        raise ValueError("FIX-3: laip (previous-period LAI) is required")
    lat, lon = domain["LAT"], domain["LON"]
    lai = np.nan_to_num(np.asarray(met["lai"], dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    laip = np.nan_to_num(np.asarray(laip, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)

    met_f, bad = fill_met_nearest(met)                                    # FIX-1
    if strict:
        assert_valid_on_vegetation(met_f, lai)
    d_temp = fill_field_nearest(d_temp, *VALID_RANGE["temp"])             # same rule for daily T
    d_ppfd = np.nan_to_num(np.asarray(d_ppfd, dtype=np.float64), nan=0.0)

    can = canopy.megcan(met_f["temp"], met_f["par"], met_f["wind"], met_f["pres"],
                        met_f["qv"], lat, lon, lai, canopy_type_fractions, day, hour_utc)
    gamsm = soil.gamma_sm(met_f["soilm"], wilt)
    er = megvea.emission_activity(can, laic=lai, laip=laip, d_temp=d_temp, d_ppfd=d_ppfd,
                                  gamsm=gamsm, nlai=nlai)                 # FIX-2, FIX-3

    gamno = None
    if date_yyyyddd is not None:
        kw = {}
        if soiltype is not None and "soilt" in met_f and met_f.get("soilt") is not None:
            kw = dict(soilt=met_f["soilt"], soilm=met_f["soilm"], soiltype=soiltype)
        cfno, cfnog = soil.soilnox(met_f["temp"], lai, lat, date_yyyyddd, **kw)
        gday, _ = soil.growseason(date_yyyyddd, lat)
        gamno = soil.gamma_no(cfno, cfnog, canopy_type_fractions, gday)

    compounds = spc.speciate_to_compounds(er, ef, gamno=gamno)
    classes = spc.compounds_to_classes(compounds)
    return {"compounds": compounds, "classes": classes, "er": er, "gamno": gamno,
            "filled": bad & (lai > 0)}


# --------------------------------------------------------------------------- FIX-5
EVT_NODATA = 65535


def fill_ecotype_nearest(src_nc: str, dst_nc: str, nodata: int = EVT_NODATA, var: str = "EVT") -> int:
    """Copy the ecotype NetCDF replacing no-data pixels by the nearest valid ecotype.

    Returns the number of pixels filled.  Ocean pixels are filled too, which is harmless because
    ocean cells carry no MapBiomas vegetation (zero growth-form fractions -> zero EF weight)."""
    from netCDF4 import Dataset
    import shutil
    shutil.copyfile(src_nc, dst_nc)
    with Dataset(dst_nc, "r+") as nc:
        evt = np.asarray(nc.variables[var][:])
        bad = evt == nodata
        if bad.any():
            ii, jj = nearest_valid_index(bad)
            evt = evt.copy(); evt[bad] = evt[ii[bad], jj[bad]]
            nc.variables[var][:] = evt
        return int(bad.sum())


# --------------------------------------------------------------------------- FIX-6
def gamtp_megan321(lai, sun_t, sha_t, sun_p, sha_p, sun_f, d_temp, d_ppfd, ldf, s, layers=5):
    """GAMTP exactly as megvea.f L725-736 (CDEA multiplies the sunlit light-dependent term only)."""
    g = _gamma
    _, vpgwt = _sp.canopy_nodes_weights(layers)
    cdea = g.gamma_cd(lai, layers)
    dt = np.asarray(d_temp, dtype=np.float64)[..., None]
    dp = np.asarray(d_ppfd, dtype=np.float64)[..., None]
    ldf_b = np.asarray(ldf, dtype=np.float64)[..., None]
    ea1 = (cdea * g.gamtld(sun_t, dt, s) * g.gamp(sun_p, dp) * sun_f
           + g.gamtld(sha_t, dt, s) * g.gamp(sha_p, dp) * (1.0 - sun_f))
    ea2 = g.gamtli(sun_t, s) * sun_f + g.gamtli(sha_t, s) * (1.0 - sun_f)
    return ((ea1 * ldf_b + ea2 * (1.0 - ldf_b)) * vpgwt).sum(axis=-1)


# --------------------------------------------------------------------------- FIX-7
def soilnox_megan321(tair, lai, lat, date_yyyyddd, *, soilt=None, soilm=None, soiltype=None,
                     precadj=1.0, corrected=True):
    """soil.soilnox with the TAIR <= 303 K cap applied before both branches, as in soilnox.F."""
    t = np.minimum(np.asarray(tair, dtype=np.float64), 315.0)
    t = np.minimum(t, 303.0)
    return _ORIG["soilnox"](t, lai, lat, date_yyyyddd, soilt=soilt, soilm=soilm, soiltype=soiltype,
                            precadj=precadj, corrected=corrected)


# --------------------------------------------------------------------------- FIX-8
def deaccumulate(acc_cur, acc_prev, hour_utc):
    """Hourly mean flux (W m-2) over (h-1, h] from ERA5-Land accumulations (J m-2) - numpy reference.

    acc_cur = field valid at hour h, acc_prev = field valid at h-1.  At 01 UTC the field already is the
    1-h accumulation; at 00 UTC it is the 24-h total ending at 00 UTC, so the previous hour (23 UTC of the
    previous day) must be subtracted, like at every other hour."""
    prev = 0.0 if int(hour_utc) == 1 else np.asarray(acc_prev, dtype=np.float64)
    return np.maximum(np.asarray(acc_cur, dtype=np.float64) - prev, 0.0) / 3600.0


def hourly_flux_fixed(ee, region, date):
    """Earth Engine version of `deaccumulate` for surface_solar_radiation_downwards (replaces era5.hourly_flux)."""
    from .gee import era5 as _era5
    d = ee.Date(date)
    col = ee.ImageCollection(_era5.COLLECTION).filterBounds(region)
    band = "surface_solar_radiation_downwards"
    cur = col.filterDate(d, d.advance(1, "hour")).first().select(band)
    prev = ee.Image(col.filterDate(d.advance(-1, "hour"), d).first()).select(band)
    prev_acc = ee.Image(ee.Algorithms.If(ee.Number(d.get("hour")).eq(1), ee.Image(0), prev))
    return cur.subtract(prev_acc).max(0).divide(3600.0).rename("SW_WM2")


_ORIG = {}
def apply_engine_patches():
    """Install FIX-6, FIX-7 and FIX-8 in the running engine (idempotent).  Returns the list of patches."""
    from .gee import era5 as _era5
    if "gamtp" not in _ORIG:
        _ORIG["gamtp"] = _gamma.gamtp
        _ORIG["soilnox"] = soil.soilnox
        _ORIG["hourly_flux"] = _era5.hourly_flux
    _gamma.gamtp = gamtp_megan321          # megvea.emission_activity calls gamma.gamtp at run time
    soil.soilnox = soilnox_megan321        # run_timestep_fixed calls soil.soilnox at run time
    _era5.hourly_flux = hourly_flux_fixed  # era5.megan_met_image calls hourly_flux at run time
    return ["FIX-6 gamtp_megan321", "FIX-7 soilnox_megan321", "FIX-8 hourly_flux_fixed"]


def patches_active() -> bool:
    from .gee import era5 as _era5
    return (_gamma.gamtp is gamtp_megan321 and soil.soilnox is soilnox_megan321
            and _era5.hourly_flux is hourly_flux_fixed)
'''
FIXES_SHA = hashlib.sha256(FIXES_PY.encode('utf-8')).hexdigest()[:16]
with open(os.path.join(MEGAN_DIR, 'src', 'megan_sc', 'fixes.py'), 'w', encoding='utf-8') as _f:
    _f.write(FIXES_PY)
import importlib
for _m in [m for m in list(sys.modules) if m.startswith('megan_sc')]:
    del sys.modules[_m]                                  # fresh import of the whole package
from megan_sc import fixes
print('fixes.py written | sha256[:16] =', FIXES_SHA)

In [ ]:
# 3 · Settings -------------------------------------------------------------------------------------------
RUN_CONFIG   = 'config/santa_catarina_2003_2025.yaml'   # <-- the run configuration (path relative to the repository)
EE_PROJECT   = None                     # <-- your Earth Engine project id; None = earth_engine.project of RUN_CONFIG
OUT_BASE     = '/content/drive/MyDrive' if IN_COLAB else os.path.dirname(REPO_DIR)   # <-- the output folder goes here
YEARS        = None                     # <-- None = every year of the period; or a few, e.g. [2019], and come back later
MONTHS       = None                     # None = every month of the period; or calendar months, e.g. [1, 4, 7, 10]
TEST_DAYS    = None                     # e.g. 2 for a quick preview (outputs get a _testNd suffix, not used downstream)
VARIANT      = 'base'                   # 'base' | 'hourly' or 'lai_eedefault' (sampling tests of the Santa Catarina example)
LAI_AGG      = 'mean'                   # 'mean': area mean of QC-valid 500 m MODIS pixels per grid cell (explicit
                                        #  reduceResolution) | 'ee_default': Earth Engine default resampling
MAKE_MAPS    = False                    # monthly and annual PNG maps (slow on Drive); notebook 25 draws the result maps
DIAG         = True                     # cheap diagnostics in the fix log (soil-NO season alternative, mean GAMSM)

In [ ]:
# 4 · Configuration, Earth Engine, engine patches and FIX-5 ------------------------------------------------
from megan_sc import domain as dom, run, landcover, stats as st, fixes
from megan_sc import gee
from megan_sc.io import geotiff, netcdf as ncio
from megan_sc.engine import speciation as spc, species as sp, soil
from megan_sc.gee import era5, lai as laimod, mapbiomas, modis_lc, region_from_bounds
def in_repo(p):
    """Absolute path of a configuration entry (relative entries are relative to the repository)."""
    return p if (not p or os.path.isabs(p)) else os.path.join(MEGAN_DIR, p)
cfg = run.load_config(in_repo(RUN_CONFIG))
for k in ('ef_table', 'speciation_dir', 'ecotype'):
    cfg['datasets']['emission_factors'][k] = in_repo(cfg['datasets']['emission_factors'][k])
if cfg['domain'].get('mode') == 'shapefile':
    cfg['domain']['shapefile'] = in_repo(cfg['domain']['shapefile'])
REG = cfg.get('region', {})
REGION_NAME = REG.get('name', cfg['run']['name']); REGION_ABBR = REG.get('abbr', cfg['run']['name'])
BOUNDARY = in_repo(REG.get('boundary') or cfg['domain'].get('shapefile'))
ZONES, ZONES_ID = in_repo(REG.get('zones')), REG.get('zones_id')
veg = cfg['datasets']['vegetation']
GF_SOURCE = veg.get('growth_form_source', 'mapbiomas')
assert GF_SOURCE in ('mapbiomas', 'modis'), f'growth_form_source {GF_SOURCE!r}: this notebook reads mapbiomas or modis'
LC_FIRST, LC_LAST = {'mapbiomas': (1985, 2024), 'modis': (2001, 2024)}[GF_SOURCE]   # years of the products in Earth Engine
LC_FIRST, LC_LAST = int(veg.get('first_year', LC_FIRST)), int(veg.get('last_year', LC_LAST))
def veg_year(y):
    """Land-cover year used for year y: y itself, or the nearest year of the land-cover product."""
    return min(max(int(y), LC_FIRST), LC_LAST)
_p0, _p1 = cfg['period']['start'], cfg['period']['end']
PERIOD_MONTHS = [(y, m) for y in range(int(_p0[:4]), int(_p1[:4]) + 1) for m in range(1, 13)
                 if _p0 <= f'{y}-{m:02d}' <= _p1]
YEARS = sorted({y for y, _ in PERIOD_MONTHS}) if YEARS is None else [y for y in YEARS if any(yy == y for yy, _ in PERIOD_MONTHS)]
SAMPLE_HOURS = int(cfg['period'].get('sample_hours', 3))   # production setting 3: 8 steps per day
if VARIANT == 'hourly':
    SAMPLE_HOURS = 1
if VARIANT == 'lai_eedefault':
    LAI_AGG = 'ee_default'
assert VARIANT in ('base', 'hourly', 'lai_eedefault'), VARIANT
MECH = cfg.get('species', {}).get('mechanism', 'CB6X')
OUT_DIR_NAME = cfg.get('output', {}).get('dir', f"megan_sc_outputs_{cfg['run']['name']}")
OUT_TAG = cfg.get('output', {}).get('tag', cfg['run']['name'])
OUT_ROOT = os.path.join(OUT_BASE, OUT_DIR_NAME + ('' if VARIANT == 'base' else f'_{VARIANT}'))
os.makedirs(OUT_ROOT, exist_ok=True)
EE_PROJECT = EE_PROJECT or cfg.get('earth_engine', {}).get('project')
ee = gee.initialize(EE_PROJECT, service_account_key=cfg.get('earth_engine', {}).get('service_account_key'))
print('Earth Engine ready:', EE_PROJECT)
PATCHES = fixes.apply_engine_patches()                  # FIX-6, FIX-7, FIX-8
assert fixes.patches_active(), 'engine patches not active'
_eco = cfg['datasets']['emission_factors']['ecotype']
_stem = os.path.splitext(os.path.basename(_eco))[0]
_eco_f = _eco if _stem.endswith('_filled') else os.path.join(os.path.dirname(_eco), _stem + '_filled.nc')   # FIX-5
if not os.path.exists(_eco_f):
    print('FIX-5: ecotype no-data pixels filled:', fixes.fill_ecotype_nearest(_eco, _eco_f))
cfg['datasets']['emission_factors']['ecotype'] = _eco_f
domain = dom.build_domain(cfg); region = region_from_bounds(domain['bounds'])
res = cfg['domain']['resolution_deg']
lon, lat, area = domain['lon'], domain['lat'], domain['area']
WILT = float(cfg['datasets'].get('soil', {}).get('constant_wilt', 0.12))
HOURS = list(range(0, 24, SAMPLE_HOURS)); SEC_PER_STEP = SAMPLE_HOURS * 3600.0
CLS = list(sp.NAMES); MET_BANDS = ['TEMP2', 'PAR', 'WINDSPD', 'PRES', 'QV', 'SOILM']
# soil-NO compound(s) for the DIAG alternative (EF_NO x GAMNO x speciation fraction)
_NO_MAP = [(int(spc.MAP_SPCA[s]), float(spc.MAP_EFFS[s])) for s in range(spc.MAP_SPCA.size) if spc.MAP_MG20[s] == sp.IDX_NO]
print(f'{REGION_NAME}: years {YEARS[0]}-{YEARS[-1]} ({len(PERIOD_MONTHS)} months in the period) | {24 // SAMPLE_HOURS} steps/day '
      f'| land cover {GF_SOURCE} {LC_FIRST}-{LC_LAST} | LAI_AGG={LAI_AGG} | -> {OUT_ROOT}')
print('patches:', PATCHES, '| grid', lat.size, 'x', lon.size, '| wilt', WILT, '| NO compounds', _NO_MAP)

In [ ]:
# 5 · Reporting setup: region mask, palettes, map helper, zones ----------------------------------------------
import geopandas as gpd
from rasterio.features import rasterize as _rasterize
from mpl_toolkits.axes_grid1 import make_axes_locatable
DPI = 300
ASPECT = 1.0 / np.cos(np.deg2rad(float(np.mean(lat))))
assert BOUNDARY and os.path.exists(BOUNDARY), f'region boundary not found: {BOUNDARY}'
_gg = gpd.read_file(BOUNDARY).to_crs('EPSG:4326')
_BND = _gg.geometry.union_all() if hasattr(_gg.geometry, 'union_all') else _gg.geometry.unary_union
_tr = geotiff._transform(lon, lat)
MASK = np.flipud(_rasterize([(_BND, 1)], out_shape=(lat.size, lon.size), transform=_tr, fill=0, dtype='uint8')).astype(bool)
domain['mask'] = MASK                                   # a cell belongs to the region when its centre lies inside
def region_clip(field): return np.where(MASK, field, np.nan)
def draw_boundary(ax, lw=0.5):
    for gm in getattr(_BND, 'geoms', [_BND]):
        try: xs, ys = gm.exterior.xy; ax.plot(xs, ys, color='0.25', lw=lw)
        except Exception: pass
FAMILIES = [('Isoprene-type', ['ISOP', 'MBO'], 'Greens'), ('Monoterpenes', ['MT_PINE', 'MT_ACYC', 'MT_CAMP', 'MT_SABI', 'MT_AROM'], 'Blues'),
            ('Sesquiterpenes', ['SQT_HR', 'SQT_LR'], 'Purples'), ('Oxygenated VOC', ['MEOH', 'ACTO', 'ETOH', 'ACID', 'OXPROD'], 'Oranges'),
            ('Light hydrocarbons', ['LVOC'], 'YlGn'), ('Other / stress VOC', ['STRESS', 'OTHER'], 'YlOrBr'),
            ('Soil NO', ['NO'], 'Reds'), ('CO', ['CO'], 'Greys')]
FAM_OF = {s: (nm, cm) for nm, ss, cm in FAMILIES for s in ss}
def save_map(field, title, path_png, cmap='viridis', vmin=None, vmax=None):
    if not MAKE_MAPS: return
    cm_ = plt.get_cmap(cmap).copy(); cm_.set_bad('white')
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    im = ax.imshow(np.ma.masked_invalid(region_clip(field)), origin='lower', extent=[lon[0], lon[-1], lat[0], lat[-1]],
                   cmap=cm_, vmin=vmin, vmax=vmax, aspect=ASPECT)
    draw_boundary(ax); ax.set_title(title); ax.axis('off')
    fig.colorbar(im, cax=make_axes_locatable(ax).append_axes('right', size='4.5%', pad=0.05))
    fig.tight_layout(); fig.savefig(path_png, dpi=DPI, bbox_inches='tight'); plt.close(fig)
_ZONE = _IDS = None
if ZONES:
    try:
        _g = gpd.read_file(ZONES).to_crs('EPSG:4326')
        _idc = ZONES_ID if ZONES_ID in _g.columns else ('CD_MUN' if 'CD_MUN' in _g.columns else _g.columns[0])
        _ZONE, _IDS = st.rasterize_municipalities(_g, lon, lat, _idc)
        print(f'zones: {len(_IDS)} polygons ({os.path.basename(ZONES)}, id {_idc})')
    except Exception as e:
        print('zones file not read (per-zone tables skipped):', e)
print(f'{REGION_ABBR} mask: {int(MASK.sum())} of {MASK.size} cells')

In [ ]:
# 6 · Earth Engine samplers: meteorology, 24-h mean temperature, LAI (FIX-4 gap fill; explicit aggregation) --
def sample_multi(img, bands):
    grid = {'dimensions': {'width': int(lon.size), 'height': int(lat.size)},
            'affineTransform': {'scaleX': res, 'shearX': 0.0, 'translateX': float(lon[0] - res / 2),
                                'shearY': 0.0, 'scaleY': -res, 'translateY': float(lat[-1] + res / 2)},
            'crsCode': 'EPSG:4326'}
    for attempt in range(4):                              # retry transient EE errors
        try:
            st_ = ee.data.computePixels({'expression': img.select(bands), 'grid': grid, 'fileFormat': 'NUMPY_NDARRAY'})
            return {b: np.flipud(np.asarray(st_[b], dtype='float64')) for b in bands}
        except Exception as e:
            if attempt == 3: raise
            print('   EE retry', attempt + 1, repr(e)[:90]); time.sleep(10 * (attempt + 1))

def daily_tmean_image(day_iso):
    """FIX-2: mean of the 24 hourly ERA5-Land 2 m temperatures of the UTC day (daymet.f: D_TEMP)."""
    d = ee.Date(day_iso)
    return (ee.ImageCollection(era5.COLLECTION).filterBounds(region).filterDate(d, d.advance(1, 'day'))
            .select('temperature_2m').mean().rename('D_TEMP'))

def lai_image(y, m):
    img = laimod.monthly_lai(ee, region, y, m).rename('LAI')
    if LAI_AGG == 'mean':                                 # area mean of the QC-valid 500 m pixels in each grid cell
        proj = ee.ImageCollection(laimod.COLLECTION).filterBounds(region).first().select('Lai').projection()
        img = img.setDefaultProjection(proj).reduceResolution(reducer=ee.Reducer.mean(), maxPixels=1024)
    return img

_lai_raw, _lai_cache, LAI_GAP_LOG = {}, {}, {}
def _shift(y, m, k):
    t = y * 12 + (m - 1) + k
    return t // 12, t % 12 + 1
def lai_raw(y, m):
    if (y, m) not in _lai_raw:
        a = np.nan_to_num(sample_multi(lai_image(y, m), ['LAI'])['LAI'], nan=0.0, posinf=0.0, neginf=0.0)
        if veg.get('lai_mask', True):
            a = a * (VEG_FRAC > 0)
        _lai_raw[(y, m)] = a
    return _lai_raw[(y, m)]
def month_lai(m, y=None):
    """FIX-4: gap-filled monthly LAI (used as LAIc, and for month-1 as LAIp)."""
    y = YEAR if y is None else y
    if (y, m) not in _lai_cache:
        yp, mp = _shift(y, m, -1); yn, mn = _shift(y, m, +1)
        filled, gap = fixes.fill_lai_gaps(lai_raw(y, m), lai_raw(yp, mp), lai_raw(yn, mn), VEG_FRAC)
        _lai_cache[(y, m)] = filled; LAI_GAP_LOG[(y, m)] = int(gap.sum())
    return _lai_cache[(y, m)]

def no_tonnes(gamno, ef_no):
    """Soil-NO tonnes for one step from a GAMNO field (same unit chain as speciation.py / mgn2mech.F)."""
    t = np.zeros_like(gamno)
    for ci, f in _NO_MAP:
        t += gamno * ef_no * f * spc.NANO2MOL * area * spc.SPCA_MWT[ci] * spc.G2TONNE * SEC_PER_STEP
    return t

In [ ]:
# 7 · One month and one year ------------------------------------------------------------------------------
def _finalize_month(m, tag, tonnes, meanflux, meanmet, diurnal):
    ncio.write_emissions(os.path.join(OUT, 'emissions', f'emis_{tag}.nc'), tonnes, CLS, lon, lat,
                         units='tonnes/month', global_attrs={'year': YEAR, 'month': m, 'mechanism': MECH,
                                                             'megan_sc_fixes_sha256': FIXES_SHA, 'variant': VARIANT})
    geotiff.write_species_stack(os.path.join(OUT, 'emissions', f'emis_{tag}.tif'), tonnes, lon, lat, CLS)
    for nm in ['ISOP', 'MT_PINE', 'SQT_HR', 'NO', 'CO']:
        save_map(tonnes[sp.index_of(nm)], f'{nm} {tag} (t/month, {REGION_ABBR})', os.path.join(OUT, 'maps', f'{nm}_{tag}.png'),
                 cmap=FAM_OF.get(nm, (None, 'YlOrRd'))[1], vmin=0)
    geotiff.write_geotiff(os.path.join(OUT, 'inputs', f'met_{tag}.tif'), meanmet, lon, lat, band_names=MET_BANDS, nodata=np.nan)
    geotiff.write_geotiff(os.path.join(OUT, 'inputs', f'lai_{tag}.tif'), month_lai(m), lon, lat, band_names=['LAI'])
    pd.DataFrame(st.species_stats_rows(tonnes, CLS, YEAR, m, mask=domain.get('mask'))).to_csv(
        os.path.join(OUT, 'stats', f'stats_{tag}.csv'), index=False)
    if _ZONE is not None:
        pd.DataFrame(st.municipality_table(tonnes, CLS, _ZONE, _IDS, YEAR, m)).to_csv(
            os.path.join(OUT, 'stats', f'municipality_{tag}.csv'), index=False)
    pd.DataFrame(diurnal, index=CLS, columns=[f'{h:02d}h' for h in HOURS]).to_csv(os.path.join(OUT, 'hourly', f'diurnal_{tag}.csv'))

def run_month(m):
    ndays_full = calendar.monthrange(YEAR, m)[1]
    ndays = ndays_full if TEST_DAYS is None else min(int(TEST_DAYS), ndays_full)
    tag = f'{YEAR}_{m:02d}' + ('' if TEST_DAYS is None else f'_test{ndays}d')
    if os.path.exists(os.path.join(OUT, 'emissions', f'emis_{tag}.nc')):
        print(f'  {tag}: already done, skipping'); return
    ck = os.path.join(OUT, 'checkpoints', f'{tag}.npz')
    steps = [(d, h) for d in range(1, ndays + 1) for h in HOURS]
    ny, nx = lat.size, lon.size
    keys = ('t', 'f', 'met', 'metn', 'fill', 'di', 'gsm', 'noalt')
    loaded = st.load_checkpoint(ck)
    if loaded is not None:
        A_, done = loaded
        acc = {k: A_[k] for k in keys}
        print(f'  {tag}: resuming ({len(done)}/{len(steps)} steps done)')
    else:
        acc = dict(t=np.zeros((len(CLS), ny, nx)), f=np.zeros((len(CLS), ny, nx)), met=np.zeros((len(MET_BANDS), ny, nx)),
                   metn=np.zeros((ny, nx)), fill=np.zeros((ny, nx)), di=np.zeros((len(CLS), len(HOURS))),
                   gsm=np.zeros((ny, nx)), noalt=np.zeros((ny, nx)))
        done = set()
    lai_c = month_lai(m)                                       # FIX-4
    yp, mp = _shift(YEAR, m, -1); lai_p = month_lai(mp, yp)     # FIX-3
    ef_no = ef[sp.IDX_NO]
    for d in range(1, ndays + 1):
        idx_day = [i for i, (dd, _) in enumerate(steps) if dd == d]
        if all(i in done for i in idx_day):
            continue
        doy = datetime.date(YEAR, m, d).timetuple().tm_yday; day_iso = f'{YEAR}-{m:02d}-{d:02d}'; date = YEAR * 1000 + doy
        mets = {h: sample_multi(era5.megan_met_image(ee, region, ee.Date(f'{day_iso}T{h:02d}:00')), MET_BANDS) for h in HOURS}
        D_TEMP = fixes.fill_field_nearest(sample_multi(daily_tmean_image(day_iso), ['D_TEMP'])['D_TEMP'], *fixes.VALID_RANGE['temp'])
        par_f = []
        for h in HOURS:
            mm_, _ = fixes.fill_met_nearest({'temp': mets[h]['TEMP2'], 'par': mets[h]['PAR'], 'wind': mets[h]['WINDSPD'],
                                             'pres': mets[h]['PRES'], 'qv': mets[h]['QV'], 'soilm': mets[h]['SOILM']})
            par_f.append(mm_['par'])
        D_PPFD = np.mean(par_f, axis=0) * fixes.PAR_WM2_TO_PPFD                                 # FIX-2
        for i in idx_day:
            if i in done:
                continue
            h = steps[i][1]; met = mets[h]
            metd = {'temp': met['TEMP2'], 'par': met['PAR'], 'wind': met['WINDSPD'], 'pres': met['PRES'],
                    'qv': met['QV'], 'soilm': met['SOILM'], 'lai': lai_c}
            r = fixes.run_timestep_fixed(domain, ef, metd, day=doy, hour_utc=h, canopy_type_fractions=ctf, d_temp=D_TEMP,
                                         d_ppfd=D_PPFD, laip=lai_p, nlai=12, wilt=WILT, date_yyyyddd=date, strict=True)
            tph = spc.compound_tonnes(spc.flux_to_mol_s(r['compounds'], area[None, :, :]), SEC_PER_STEP)
            acc['t'] += spc.compounds_to_classes(tph); acc['f'] += r['classes']; acc['fill'] += r['filled']
            land = np.isfinite(met['TEMP2']) & (met['TEMP2'] > 150) & (met['TEMP2'] < 340)   # input maps stay land-only
            acc['met'] += np.stack([np.where(land, np.nan_to_num(metd[k]), 0.0) for k in ['temp', 'par', 'wind', 'pres', 'qv', 'soilm']])
            acc['metn'] += land
            acc['di'][:, HOURS.index(h)] += np.nansum(r['classes'], axis=(1, 2))
            if DIAG:
                mf, _ = fixes.fill_met_nearest({k: metd[k] for k in ('temp', 'par', 'wind', 'pres', 'qv', 'soilm')})
                acc['gsm'] += soil.gamma_sm(mf['soilm'], WILT)
                cfno, cfnog = soil.soilnox(mf['temp'], lai_c, domain['LAT'], date, corrected=False)   # MEGANv3.21 SH season
                g_alt = soil.gamma_no(cfno, cfnog, ctf, soil.growseason(date, domain['LAT'], corrected=False)[0])
                acc['noalt'] += no_tonnes(g_alt, ef_no)
            done.add(i)
            st.save_checkpoint(ck, acc, done)
        if d % 5 == 0 or d == ndays:
            print(f'    {tag} day {d}/{ndays}')
    n = len(steps)
    meanmet = acc['met'] / np.where(acc['metn'] > 0, acc['metn'], np.nan)
    _finalize_month(m, tag, acc['t'], acc['f'] / n, meanmet, acc['di'] / ndays)
    np.savez_compressed(os.path.join(OUT, 'inputs', f'fixlog_{tag}.npz'), filled_met_steps=acc['fill'],
                        lai_gap_cells=np.array(LAI_GAP_LOG.get((YEAR, m), 0)), gamsm_mean=acc['gsm'] / n,
                        no_tonnes_mgn321_season=acc['noalt'], steps=np.array(n))
    os.remove(ck)
    print(f'  {tag}: done | vegetated cells with filled met: {int((acc["fill"] > 0).sum())} | LAI gap cells filled: {LAI_GAP_LOG.get((YEAR, m), 0)}')

def growth_forms(year):
    """Growth-form fractions (tree, shrub, herb, crop, veg) of a land-cover year on the grid, from Earth Engine."""
    if GF_SOURCE == 'mapbiomas':
        cid = mapbiomas.resolve_collection_id(ee, year, veg.get('collection'))
        return mapbiomas.growth_form_fractions_gridded(ee, region, domain, year, collection_id=cid), cid
    img = modis_lc.growth_form_fractions(ee, region, year, out_scale_m=res * 111000.0)
    bands = list(landcover.GROWTH_FORMS) + ['veg']
    g = mapbiomas.sample_image_tiled(ee, img, domain, bands, ntiles=4)
    return {b: np.nan_to_num(g[b], nan=0.0) for b in bands}, None

def setup_year(y):
    """Per-year globals: output folders, growth forms (cached), EF grid, canopy fractions."""
    global YEAR, VEG_YEAR, OUT, gf_np, ef, ctf, VEG_FRAC
    YEAR = y; VEG_YEAR = veg_year(y); OUT = os.path.join(OUT_ROOT, str(y))
    for sub in ('inputs', 'emissions', 'maps', 'stats', 'hourly', 'checkpoints'):
        os.makedirs(os.path.join(OUT, sub), exist_ok=True)
    gf_cache = os.path.join(OUT, 'inputs', f'growthform_{VEG_YEAR}.npz')
    cid = None
    if os.path.exists(gf_cache):
        gf_np = {k: v for k, v in np.load(gf_cache).items()}
    else:
        print(f'  {GF_SOURCE} growth forms of {VEG_YEAR} (tiled in-memory pull, ~1-2 min)...')
        gf_np, cid = growth_forms(VEG_YEAR); np.savez(gf_cache, **gf_np)
    ef = run.build_ef_grid_multiform(cfg, domain, gf_np)
    np.savez_compressed(os.path.join(OUT, 'inputs', f'ef_{y}.npz'), ef=ef.astype('float32'), classes=np.array(CLS))   # for notebook 11 (C3)
    ctf = landcover.canopy_type_fractions(gf_np, veg['tree_canopy_type'])
    VEG_FRAC = sum(gf_np[g] for g in landcover.GROWTH_FORMS)
    geotiff.write_geotiff(os.path.join(OUT, 'inputs', f'growthform_{VEG_YEAR}.tif'),
                          np.stack([gf_np[g] for g in landcover.GROWTH_FORMS]), lon, lat, band_names=list(landcover.GROWTH_FORMS))
    json.dump(dict(year=y, variant=VARIANT, sample_hours=SAMPLE_HOURS, lai_aggregation=LAI_AGG, wilt=WILT,
                   fixes_sha256=FIXES_SHA, patches=PATCHES, ecotype=_eco_f, engine=MEGAN_DIR, ee_project=EE_PROJECT,
                   config=RUN_CONFIG, region=REGION_NAME, growth_form_source=GF_SOURCE, land_cover_year=VEG_YEAR,
                   mapbiomas_collection_id=cid, started=datetime.datetime.utcnow().isoformat() + 'Z'),
              open(os.path.join(OUT, 'provenance_v10.json'), 'w'), indent=1)

def annual_rollup(y):
    """Annual sum of the 12 monthly files and the totals of the region (skipped when a month is missing)."""
    from netCDF4 import Dataset
    mfiles = [f for f in sorted(glob.glob(os.path.join(OUT, 'emissions', f'emis_{y}_*.nc'))) if 'test' not in os.path.basename(f)]
    if len(mfiles) < 12:
        print(f'  {y}: {len(mfiles)} months finished, annual roll-up skipped'); return None
    annual = {c: np.zeros((lat.size, lon.size)) for c in CLS}
    for f in mfiles:
        with Dataset(f) as ds:
            for c in CLS:
                a = np.array(ds.variables[c][:], dtype='float64'); annual[c] += np.where(np.isfinite(a) & (a > -9990), a, 0.0)
    stack = np.stack([annual[c] for c in CLS])
    ncio.write_emissions(os.path.join(OUT, 'emissions', f'annual_emis_{y}.nc'), stack, CLS, lon, lat, units='tonnes/year',
                         global_attrs={'year': y, 'months': '1-12', 'mechanism': MECH, 'megan_sc_fixes_sha256': FIXES_SHA, 'variant': VARIANT})
    geotiff.write_species_stack(os.path.join(OUT, 'emissions', f'annual_emis_{y}.tif'), stack, lon, lat, CLS)
    tot = pd.DataFrame([{'species': c, f'annual_tonnes_{REGION_ABBR}': float(np.nansum(region_clip(annual[c]))), 'family': FAM_OF.get(c, ('', ''))[0]} for c in CLS])
    tot.to_csv(os.path.join(OUT, 'stats', f'annual_by_family_{y}.csv'), index=False)
    return tot.set_index('species')[f'annual_tonnes_{REGION_ABBR}']

In [ ]:
# 8 · Run (safe to re-run after any interruption) -----------------------------------------------------------
summary = {}
for y in YEARS:
    t0 = time.time(); print(f'=== {y} ===')
    setup_year(y); _lai_cache.clear(); LAI_GAP_LOG.clear()
    for m in [mm for yy, mm in PERIOD_MONTHS if yy == y and (MONTHS is None or mm in MONTHS)]:
        run_month(m)
    tot = annual_rollup(y)
    if tot is not None:
        summary[y] = tot
        print(f'  {y} {REGION_ABBR} totals (kt): BVOC {tot.drop(["NO", "CO"]).sum() / 1e3:.1f} | ISOP {tot["ISOP"] / 1e3:.1f} | '
              f'SQT {(tot["SQT_HR"] + tot["SQT_LR"]) / 1e3:.1f} | NO {tot["NO"] / 1e3:.2f} | {(time.time() - t0) / 60:.0f} min')
    _lai_raw.clear()
if summary:
    pd.DataFrame(summary).T.to_csv(os.path.join(OUT_ROOT, f'annual_totals_{OUT_TAG}_{min(summary)}_{max(summary)}.csv'))

In [ ]:
# 9 · Quick self-check of the years just run (the full check is notebook 11) -----------------------------------
rows = []
for y in summary:
    from netCDF4 import Dataset
    with Dataset(os.path.join(OUT_ROOT, str(y), 'emissions', f'annual_emis_{y}.nc')) as ds:
        E = {c: np.nan_to_num(np.array(ds.variables[c][:], dtype='float64')) for c in CLS}
    bv = sum(E[c] for c in CLS if c not in ('NO', 'CO'))
    tree = np.load(os.path.join(OUT_ROOT, str(y), 'inputs', f'growthform_{veg_year(y)}.npz'))['tree']
    sig = MASK & (tree > 0.1) & (bv > 0) & (E['ISOP'] == 0)                  # the 340 K / PAR = 0 signature
    logs = sorted(glob.glob(os.path.join(OUT_ROOT, str(y), 'inputs', f'fixlog_{y}_*.npz')))
    filled = sum(int((np.load(f)['filled_met_steps'] > 0).sum()) for f in logs)
    rows.append(dict(year=y, signature_cells=int(sig.sum()), fixlogs=len(logs), filled_cell_months=filled))
print(pd.DataFrame(rows).to_string(index=False))
print('signature_cells must be 0 in every year; fixlogs must be 12.')